In [17]:
# ---------------------------------------------------------
# Comparison:
# Custom Transformer vs Hugging Face Transformer
# ---------------------------------------------------------

import torch
import torch.nn as nn
import torch.nn.functional as F

# Hugging Face
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# =========================================================
# PART 1 : CUSTOM GPT STYLE TRANSFORMER
# =========================================================

print("\n================ CUSTOM GPT MODEL ================\n")

# ---------------------------------------------------------
# Small Dataset
# ---------------------------------------------------------
text = """
deep learning is powerful
deep learning uses neural networks
transformers are powerful models
pytorch makes deep learning easy
gpt generates meaningful text
"""

# ---------------------------------------------------------
# Tokenization
# ---------------------------------------------------------
words = text.split()

vocab = sorted(set(words))

stoi = {word: i for i, word in enumerate(vocab)}
itos = {i: word for word, i in stoi.items()}

vocab_size = len(vocab)

data = [stoi[word] for word in words]

# ---------------------------------------------------------
# Hyperparameters
# ---------------------------------------------------------
d_model = 32
num_heads = 4
num_layers = 2
block_size = 4
epochs = 300
lr = 0.001

# ---------------------------------------------------------
# Training Data
# ---------------------------------------------------------
X_train = []
Y_train = []

for i in range(len(data) - block_size):

    X_train.append(data[i:i+block_size])
    Y_train.append(data[i+1:i+block_size+1])

X_train = torch.tensor(X_train)
Y_train = torch.tensor(Y_train)

# ---------------------------------------------------------
# Decoder Block
# ---------------------------------------------------------
class DecoderBlock(nn.Module):

    def __init__(self):

        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            batch_first=True
        )

        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.ReLU(),
            nn.Linear(4*d_model, d_model)
        )

    def forward(self, x):

        seq_len = x.size(1)

        # Causal Mask
        mask = torch.triu(
            torch.ones(seq_len, seq_len),
            diagonal=1
        ).bool()

        attn_output, _ = self.attn(
            x, x, x,
            attn_mask=mask
        )

        x = self.ln1(x + attn_output)

        ff_output = self.ff(x)

        x = self.ln2(x + ff_output)

        return x

# ---------------------------------------------------------
# GPT Model
# ---------------------------------------------------------
class MiniGPT(nn.Module):

    def __init__(self):

        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.position_embedding = nn.Embedding(
            block_size,
            d_model
        )

        self.blocks = nn.Sequential(*[
            DecoderBlock()
            for _ in range(num_layers)
        ])

        self.lm_head = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(self, x):

        B, T = x.shape

        tok_emb = self.token_embedding(x)

        positions = torch.arange(T)

        pos_emb = self.position_embedding(positions)

        x = tok_emb + pos_emb

        x = self.blocks(x)

        logits = self.lm_head(x)

        return logits

# ---------------------------------------------------------
# Train Custom Model
# ---------------------------------------------------------
torch.manual_seed(42)

custom_model = MiniGPT()

optimizer = torch.optim.Adam(
    custom_model.parameters(),
    lr=lr
)

for epoch in range(epochs):

    logits = custom_model(X_train)

    loss = F.cross_entropy(
        logits.view(-1, vocab_size),
        Y_train.view(-1)
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# ---------------------------------------------------------
# Generate Text
# ---------------------------------------------------------
def generate_custom(start_text, max_new_tokens=6):

    custom_model.eval()

    context = [stoi[w] for w in start_text.split()]

    for _ in range(max_new_tokens):

        x = torch.tensor([context[-block_size:]])

        logits = custom_model(x)

        logits = logits[:, -1, :]

        probs = F.softmax(logits, dim=-1)

        next_token = torch.argmax(probs).item()

        context.append(next_token)

    generated = [itos[i] for i in context]

    return " ".join(generated)

print("\nCustom Model Output:\n")

print(generate_custom("deep learning"))

# =========================================================
# PART 2 : HUGGING FACE GPT-2
# =========================================================

print("\n================ HUGGING FACE GPT-2 ================\n")

# ---------------------------------------------------------
# Load pretrained GPT-2
# ---------------------------------------------------------
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

hf_model = GPT2LMHeadModel.from_pretrained("gpt2")

# ---------------------------------------------------------
# Input Text
# ---------------------------------------------------------
input_text = "Deep learning"

inputs = tokenizer.encode(
    input_text,
    return_tensors="pt"
)

# ---------------------------------------------------------
# Generate Text
# ---------------------------------------------------------
outputs = hf_model.generate(
    inputs,
    max_length=20,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.7
)

generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Hugging Face GPT-2 Output:\n")

print(generated_text)

# =========================================================
# COMPARISON
# =========================================================

print("\n================ COMPARISON ================\n")

print("1. Custom Model")
print("- Built from scratch")
print("- Small dataset")
print("- Educational implementation")
print("- Limited vocabulary")
print("- Learns tiny patterns")

print("\n2. Hugging Face GPT-2")
print("- Pretrained on huge internet data")
print("- Better sentence generation")
print("- Large vocabulary")
print("- Production ready")
print("- Highly optimized")


================ CUSTOM GPT MODEL ================

Epoch 0, Loss: 2.9776
Epoch 50, Loss: 0.5395
Epoch 100, Loss: 0.2317
Epoch 150, Loss: 0.1693
Epoch 200, Loss: 0.1462
Epoch 250, Loss: 0.1348

Custom Model Output:

deep learning uses neural networks transformers are powerful

================ HUGGING FACE GPT-2 ================



tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Hugging Face GPT-2 Output:

Deep learning

A high-level example of ACH is the concept of "high-level

================ COMPARISON ================

1. Custom Model
- Built from scratch
- Small dataset
- Educational implementation
- Limited vocabulary
- Learns tiny patterns

2. Hugging Face GPT-2
- Pretrained on huge internet data
- Better sentence generation
- Large vocabulary
- Production ready
- Highly optimized
